In [2]:
# import bibliotek
import pandas as pd
import numpy as np

In [3]:
# wczytanie plików
account_info = pd.read_csv("data/da_fitly_account_info.csv")
customer_support = pd.read_csv("data/da_fitly_customer_support.csv")
user_activity = pd.read_csv("data/da_fitly_user_activity.csv")

In [4]:
# utworzenie schowka do kolejnych kontroli
datasets = {
    "account_info": account_info,
    "customer_support": customer_support,
    "user_activity": user_activity
}

In [6]:
# wyświetlenie tabel
for name, df in datasets.items():
    print(f"{name.upper()}")
    display(df.head())

ACCOUNT_INFO


,customer_id,email,state,plan,plan_list_price,churn_status
0,C10000,user10000@example.com,New Jersey,Enterprise,105,Y
1,C10001,user10001@example.net,Louisiana,Basic,22,Y
2,C10002,user10002@example.net,Oklahoma,Basic,24,NaN
3,C10003,user10003@example.com,Michigan,Free,0,NaN
4,C10004,user10004@example.com,Texas,Enterprise,119,NaN


CUSTOMER_SUPPORT


,ticket_time,user_id,channel,topic,resolution_time_hours,state,comments
0,2025-06-13 05:55:17.154573,10125,chat,technical,11.48,1,NaN
1,2025-08-06 13:21:54.539551,10109,chat,account,1.01,0,NaN
2,2025-08-22 12:39:35.718663,10149,chat,technical,10.09,0,Erase my data from your systems.
3,2025-06-07 02:49:46.986055,10268,phone,account,9.10,1,NaN
4,2025-07-25 00:24:38.945079,10041,phone,other,2.28,1,NaN


USER_ACTIVITY


,event_time,user_id,event_type
0,2025-09-08 15:05:39.422721,10118,watch_video
1,2025-09-08 08:15:05.264103,10220,watch_video
2,2025-11-14 06:28:35.207671,10009,share_workout
3,2025-08-20 16:53:38.682901,10227,read_article
4,2025-07-24 16:47:31.728422,10123,track_workout


In [7]:
#liczba rekodrów i kolumn
for name, df in datasets.items():
    print(f"{name}:")
    print(f"Liczba rekordów: {df.shape[0]}")
    print(f"Liczba kolumn: {df.shape[1]}")
    print()

account_info:
Liczba rekordów: 400
Liczba kolumn: 6

customer_support:
Liczba rekordów: 918
Liczba kolumn: 7

user_activity:
Liczba rekordów: 445
Liczba kolumn: 3



In [8]:
dataset_overview = pd.DataFrame({
    name: {
        "rows": df.shape[0],
        "columns": df.shape[1]
    }
    for name, df in datasets.items()
}).T

display(dataset_overview)

,rows,columns
account_info,400,6
customer_support,918,7
user_activity,445,3


The analysis uses three datasets: account information (400 records, 6 columns), customer support (918 records, 7 columns), and user activity (445 records, 3 columns).

In [10]:
# sprawdzenie nazw kolumn
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.columns.tolist())


ACCOUNT_INFO
['customer_id', 'email', 'state', 'plan', 'plan_list_price', 'churn_status']

CUSTOMER_SUPPORT
['ticket_time', 'user_id', 'channel', 'topic', 'resolution_time_hours', 'state', 'comments']

USER_ACTIVITY
['event_time', 'user_id', 'event_type']


- w account_info występuje churn_status, a nie churn;
- w customer_support występuje dodatkowa kolumna comments.
rozbieżności między plikami a tabelą z dokumerntacją

In [11]:
# sprawdzenie typów danych
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.dtypes)


ACCOUNT_INFO
customer_id        object
email              object
state              object
plan               object
plan_list_price     int64
churn_status       object
dtype: object

CUSTOMER_SUPPORT
ticket_time               object
user_id                    int64
channel                   object
topic                     object
resolution_time_hours    float64
state                      int64
comments                  object
dtype: object

USER_ACTIVITY
event_time    object
user_id        int64
event_type    object
dtype: object


Kolumny ticket_time i event_time są początkowo wczytywane jako object, mimo że reprezentują daty.

In [12]:
# sprawdzanie brakujących wartości
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    missing = df.isna().sum()
    display(missing.to_frame(name="missing_count"))


ACCOUNT_INFO


,missing_count
customer_id,0
email,0
state,0
plan,0
plan_list_price,0
churn_status,286



CUSTOMER_SUPPORT


,missing_count
ticket_time,0
user_id,0
channel,0
topic,0
resolution_time_hours,0
state,0
comments,872



USER_ACTIVITY


,missing_count
event_time,0
user_id,0
event_type,0


Missing values were identified in churn_status and comments.

In [13]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    unique_counts = df.nunique(dropna=True)
    display(unique_counts.to_frame(name="unique_values"))


ACCOUNT_INFO


,unique_values
customer_id,400
email,400
state,50
plan,4
plan_list_price,106
churn_status,1



CUSTOMER_SUPPORT


,unique_values
ticket_time,918
user_id,367
channel,4
topic,4
resolution_time_hours,735
state,2
comments,19



USER_ACTIVITY


,unique_values
event_time,445
user_id,246
event_type,4


customer_id ma 400 unikalnych wartości przy 400 rekordach. nie występują powtarzające się identyfikatory klientów.
churn_status ma tylko jedną unikalną niepustą wartość — Y. Pozostałe wartości są brakujące. Czy to oznacza, że brakującve - nie zrezygnował?

In [14]:
# sprawdzenie rozkładu zmiennych, które nas interesują ze wzgledu na badanie
# ACCOUNT_INFO
# plan subskrypcyjny
account_info["plan"].value_counts(dropna=False)

plan
Basic         118
Free          105
Enterprise     92
Pro            85
Name: count, dtype: int64

In [17]:
# stan
account_info["state"].value_counts(dropna=False)

state
Virginia          16
Vermont           14
Florida           14
Delaware          13
Colorado          12
Arizona           12
Montana           12
Idaho             11
Rhode Island      10
Alaska            10
North Dakota      10
Missouri          10
West Virginia      9
Pennsylvania       9
Ohio               9
Texas              9
Michigan           9
Massachusetts      9
Tennessee          9
Mississippi        9
Wisconsin          8
Washington         8
Utah               8
Hawaii             8
Connecticut        8
New Mexico         7
Minnesota          7
Illinois           7
Iowa               7
New York           7
Oklahoma           6
Louisiana          6
New Jersey         6
Kansas             6
South Carolina     6
Maryland           6
North Carolina     6
Nebraska           6
New Hampshire      6
Arkansas           6
Wyoming            6
Nevada             6
Indiana            6
Kentucky           5
Oregon             5
South Dakota       5
Maine              4
Califor

In [19]:
# churn_status
account_info["churn_status"].value_counts(dropna=False)

churn_status
NaN    286
Y      114
Name: count, dtype: int64

In [28]:
# cena subskrypcji
account_info["plan_list_price"].describe()

count    400.000000
mean      43.965000
std       45.263348
min        0.000000
25%        0.000000
50%       26.000000
75%       77.250000
max      148.000000
Name: plan_list_price, dtype: float64

In [20]:
#CUSTOMER_STATUS
#kanał kontaktu
customer_support["channel"].value_counts(dropna=False)

channel
email    298
chat     294
phone    287
-         39
Name: count, dtype: int64

 są "-" które wskazuja na braki w danych wcześniej niewykryte

In [22]:
# temat zgłoszenia
customer_support["topic"].value_counts(dropna=False)

topic
billing      239
other        228
technical    226
account      225
Name: count, dtype: int64

In [24]:
# state
customer_support["state"].value_counts(dropna=False)

state
1    504
0    414
Name: count, dtype: int64

Brak informacji co oznacza 0 i 1 w state.

In [31]:
# czas zgłoszeń
customer_support["resolution_time_hours"].describe()

count    918.000000
mean      10.391362
std        7.079888
min        0.520000
25%        5.112500
50%        9.040000
75%       13.137500
max       32.460000
Name: resolution_time_hours, dtype: float64

In [35]:
# sprawdzanie zakresu dat na nowej, tymczasowej kolumnie
ticket_dates = pd.to_datetime(
    customer_support["ticket_time"],
    errors="coerce"
)
print(ticket_dates.min())
print(ticket_dates.max())
print(ticket_dates.isna().sum())

2025-06-05 15:32:33.005817
2025-12-01 22:01:58.485299
0


In [ ]:
# USER ACTIVITY

In [26]:
# rodzaj zdarzenia
user_activity["event_type"].value_counts(dropna=False)

event_type
read_article     125
watch_video      120
track_workout    108
share_workout     92
Name: count, dtype: int64

In [49]:
# sprawdzanie zakresu dat na nowej, tymczasowej kolumnie
user_activity2 = pd.to_datetime(
    user_activity["event_time"],
    errors="coerce"

)
print(user_activity2.min())
print(user_activity2.max())
print(user_activity2.isna().sum())

2025-06-05 10:14:53.039663
2025-12-01 21:12:13.342817
0


The plan column contains four categories: Free, Basic, Pro, and Enterprise, consistent with the product documentation. No missing values were identified.
The churn_status column contains 114 values recorded as Y and 286 missing values. The meaning of missing churn statuses requires further investigation. W przypadku customer_support["channel"] występowanie 39 wartości -.
The numerical columns were checked for their minimum, maximum, and descriptive statistics.
No negative values were identified in the plan_list_price or resolution_time_hours columns.
The ticket_time and event_time columns were successfully converted to datetime format for validation purposes.
Both datasets contain records from June to December 2025.
No invalid date values were identified.

In [40]:
#sprawdzenie duplikatów
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    duplicated_sum = df.duplicated().sum()
    print("Liczba duplikatów:", duplicated_sum)


ACCOUNT_INFO
Liczba duplikatów: 0

CUSTOMER_SUPPORT
Liczba duplikatów: 0

USER_ACTIVITY
Liczba duplikatów: 0


In [41]:
# sprawdzenie unilalności identyfikatorów
account_info["customer_id"].duplicated().sum()

np.int64(0)

In [42]:
#sprawdzenie zduplikowanych zgłoszeń
customer_support.duplicated(subset=["user_id", "ticket_time"]).sum()

np.int64(0)

In [50]:
user_activity.duplicated(subset=["user_id", "event_time", "event_type"]).sum()

np.int64(0)

Nie wykryto powtarzających się kombinacji identyfikatora klienta i czasu zgłoszenia, jednak nie można tego wykluczyć ze 100% pewnością, bo nie ma żadnych jednoznacznych identyfikatorów.

Duplicate Validation
All three datasets were checked for duplicate records.
No exact duplicate rows were identified in any dataset.
The customer_id column in account_info contains 400 unique identifiers, confirming that each customer is represented by a single record.
No duplicate combinations of user_id and ticket_time were identified in customer_support.
No duplicate combinations of user_id, event_time and event_type were identified in user_activity.
No records were removed during duplicate validation.
However, the absence of unique ticket and event identifiers prevents us from definitively ruling out all logical duplicates.

In [ ]:
# Sprawdzenie zgodności identyfikatorów między zbiorami
# czy użytkowników występujących w customer_support i user_activity można odnaleźć w tabeli account_info, po to aby je połączyć

In [51]:
account_info["customer_id"].head()

0    C10000
1    C10001
2    C10002
3    C10003
4    C10004
Name: customer_id, dtype: object

In [55]:
customer_support["user_id"].sort_values().head()

80     10000
337    10000
595    10000
56     10001
472    10001
Name: user_id, dtype: int64

In [56]:
user_activity["user_id"].sort_values().head()

60     10000
224    10000
112    10000
39     10001
161    10002
Name: user_id, dtype: int64

Ujednolicamy zapis, tj dodajemy C z datasetów customer support oraz user activity

In [58]:
support_id = "C" + customer_support["user_id"].astype(str)

In [59]:
activity_id = "C" + user_activity["user_id"].astype(str)

nie zmieniamy oryginalnych danych. Tworzymy jedynie dwie dodatkowe zmienne, które wykorzystamy do porównania identyfikatorów.

In [63]:
# sprawdzamy czy identifikatory wszystkie w tych dwóch datasetach występują w głównym zbiorze
(~support_id.isin(account_info["customer_id"])).sum()

np.int64(0)

In [62]:
(~activity_id.isin(account_info["customer_id"])).sum()

np.int64(0)

In [64]:
# i jeszcze czy są klienci bez zgłoszeń i aktywności
(~account_info["customer_id"].isin(support_id)).sum()

np.int64(33)

In [65]:
(~account_info["customer_id"].isin(activity_id)).sum()

np.int64(154)

Nie oznacza jednak automatycznie, że 154 klientów nigdy nie korzystało z aplikacji. W danych mogą występować luki, ponieważ Lead Engineer potwierdza, że awarie systemów mogą prowadzić do brakujących rekordów.

Identifier Consistency
Customer identifiers were checked across all three datasets.
The account_info dataset uses identifiers with a "C" prefix, while customer_support and user_activity use numerical identifiers.
Identifiers were temporarily standardized to allow comparison.
All records from customer_support and user_activity could be matched to existing customer accounts.
However, 33 customers have no recorded support tickets, and 154 customers have no recorded activity.
No records were removed during this validation.